In [80]:
import pandas as pd
import numpy as np
import os

# 1. LOAD DATA
print("Files in raw folder:", os.listdir('../data/raw/'))
movies = pd.read_csv('../data/raw/movies.csv')
ratings = pd.read_csv('../data/raw/ratings.csv')

print("\n--- Movies Data ---")
display(movies.head())
print("\n--- Ratings Data ---")
display(ratings.head())

# 2. DIAGNOSTIC
print("\n" + "="*60)
print("DIAGNOSTIC REPORT")
print("="*60)

print(f"Movies shape: {movies.shape}")
print(f"Ratings shape: {ratings.shape}")
print(f"\nMovies — NaN:\n{movies.isnull().sum()}")
print(f"\nRatings — NaN:\n{ratings.isnull().sum()}")
print(f"\nMovies — duplicated title: {movies['title'].duplicated().sum()}")
print(f"Ratings — duplicated (user,movie): {ratings.duplicated(subset=['userId','movieId']).sum()}")
print(f"\nRatings — rating range: [{ratings['rating'].min()}, {ratings['rating'].max()}]")

# Backup
movies_raw = movies.copy()
ratings_raw = ratings.copy()

Files in raw folder: ['movies.csv', 'ratings.csv']

--- Movies Data ---


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



--- Ratings Data ---


,userId,movieId,rating,timestamp
0,1,16,4.0,1217897793
1,1,24,1.5,1217895807
2,1,32,4.0,1217896246
3,1,47,4.0,1217896556
4,1,50,4.0,1217896523



DIAGNOSTIC REPORT
Movies shape: (10329, 3)
Ratings shape: (105339, 4)

Movies — NaN:
movieId    0
title      0
genres     0
dtype: int64

Ratings — NaN:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Movies — duplicated title: 2
Ratings — duplicated (user,movie): 0

Ratings — rating range: [0.5, 5.0]


In [81]:
print("\n" + "="*60)
print("CLEANING")
print("="*60)

# 1. Missing values
n_before = len(ratings)
ratings = ratings.dropna(subset=['userId','movieId','rating'])
print(f"Ratings NaN حُذفت: {n_before - len(ratings)}")

movies = movies.dropna(subset=['movieId','title'])
movies['genres'] = movies['genres'].fillna('Unknown')
# إصلاح (no genres listed)
movies['genres'] = movies['genres'].str.replace('(no genres listed)', 'Unknown', regex=False)

# 2. Timestamp
ratings['timestamp'] = pd.to_datetime(ratings['timestamp'], unit='s')
ratings['year']  = ratings['timestamp'].dt.year
ratings['month'] = ratings['timestamp'].dt.month

# 3. Duplicates
n_before = len(movies)
movies = movies.drop_duplicates(subset='title', keep='first').reset_index(drop=True)
print(f"Movies — تكرار عنوان حُذف: {n_before - len(movies)}")

valid_ids = set(movies['movieId'])
ratings = ratings[ratings['movieId'].isin(valid_ids)].copy()

n_before = len(ratings)
ratings = ratings.drop_duplicates(subset=['userId','movieId'], keep='last')
print(f"Ratings — تكرار (user,movie) حُذف: {n_before - len(ratings)}")

# 4. Rating Range
n_before = len(ratings)
ratings = ratings[(ratings['rating'] >= 0.5) & (ratings['rating'] <= 5.0)]
print(f"Ratings — خارج [0.5,5] حُذفت: {n_before - len(ratings)}")


CLEANING
Ratings NaN حُذفت: 0
Movies — تكرار عنوان حُذف: 2
Ratings — تكرار (user,movie) حُذف: 0
Ratings — خارج [0.5,5] حُذفت: 0


In [82]:
print("\n" + "="*60)
print("FILTERING")
print("="*60)

MIN_USER_RATINGS  = 5
MIN_MOVIE_RATINGS = 5

# 1. فلتر المستخدمين
user_counts = ratings['userId'].value_counts()
before = (ratings['userId'].nunique(), len(ratings))
ratings = ratings[ratings['userId'].isin(user_counts[user_counts >= MIN_USER_RATINGS].index)]
print(f"Users: {before[0]} → {ratings['userId'].nunique()}  (حُذف {before[0]-ratings['userId'].nunique()} مستخدم)")

# 2. فلتر الأفلام (Flag approach)
movie_counts = ratings['movieId'].value_counts()
ratings['is_reliable_movie'] = ratings['movieId'].map(lambda m: movie_counts.get(m, 0) >= MIN_MOVIE_RATINGS)
movies['is_reliable'] = movies['movieId'].map(lambda m: movie_counts.get(m, 0) >= MIN_MOVIE_RATINGS).fillna(False)

# 3. مزامنة Movies (حذف الأفلام بدون تقييمات)
movies = movies[movies['movieId'].isin(set(ratings['movieId']))].copy()
print(f"Movies: {ratings['movieId'].nunique()} (بلا حذف من الموثوقة، فقط حذف غير المقيّمة)")


FILTERING
Users: 668 → 668  (حُذف 0 مستخدم)
Movies: 10323 (بلا حذف من الموثوقة، فقط حذف غير المقيّمة)


In [83]:
print("\n" + "="*60)
print("OUTLIERS TREATMENT (IQR Method)")
print("="*60)

# 1. Outliers in User Activity
user_counts = ratings['userId'].value_counts()
Q1, Q3 = user_counts.quantile(0.25), user_counts.quantile(0.75)
IQR = Q3 - Q1
upper_user = Q3 + 1.5 * IQR

ratings['is_power_user'] = ratings['userId'].map(lambda u: user_counts.get(u, 0) > upper_user)
print(f"Power users (نشاط زائد): {ratings['is_power_user'].sum()} تقييم ({ratings['is_power_user'].mean()*100:.1f}%)")

# 2. Outliers in Movie Popularity
movie_counts = ratings['movieId'].value_counts()
Q1m, Q3m = movie_counts.quantile(0.25), movie_counts.quantile(0.75)
IQRm = Q3m - Q1m
upper_movie = Q3m + 1.5 * IQRm

ratings['is_blockbuster'] = ratings['movieId'].map(lambda m: movie_counts.get(m, 0) > upper_movie)
print(f"Blockbusters (شعبية زائدة): {ratings['is_blockbuster'].sum()} تقييم ({ratings['is_blockbuster'].mean()*100:.1f}%)")

# 3. Flat Raters (Outliers in behavior)
user_std = ratings.groupby('userId')['rating'].std().fillna(0)
flat_raters = user_std[user_std < 0.5].index
ratings['is_flat_rater'] = ratings['userId'].isin(flat_raters)
print(f"Flat Raters (تقييم ثابت): {ratings['is_flat_rater'].sum()} تقييم ({ratings['is_flat_rater'].mean()*100:.1f}%)")
print("\n✅ Outliers: تم فحصها وإضافة flags (بلا حذف — القرار الصحيح لأنظمة التوصية)")


OUTLIERS TREATMENT (IQR Method)
Power users (نشاط زائد): 52711 تقييم (50.0%)
Blockbusters (شعبية زائدة): 71843 تقييم (68.2%)
Flat Raters (تقييم ثابت): 453 تقييم (0.4%)

✅ Outliers: تم فحصها وإضافة flags (بلا حذف — القرار الصحيح لأنظمة التوصية)


In [84]:
print("\n" + "="*60)
print("FEATURE ENGINEERING & MERGE")
print("="*60)

# 1. Genres One-Hot
genres_dummies = movies['genres'].str.get_dummies(sep='|')
genres_dummies.columns = ['genre_' + c for c in genres_dummies.columns]

# 2. Stats
movie_stats = (ratings.groupby('movieId')['rating']
               .agg(avg_rating='mean', num_ratings='count')
               .reset_index())

user_stats = (ratings.groupby('userId')['rating']
              .agg(user_avg='mean', user_count='count')
              .reset_index())

# 3. Table 1: ratings_clean
ratings_clean = ratings.merge(user_stats, on='userId', how='left')

# 4. Table 2: movies_clean
movies_clean = pd.concat(
    [movies.reset_index(drop=True), genres_dummies.reset_index(drop=True)], axis=1
).merge(movie_stats, on='movieId', how='left')

# 5. Table 3: final_df (Merged)
final_df = (ratings_clean
            .merge(movies_clean, on='movieId', how='left')
            .drop(columns=['is_reliable'], errors='ignore')) # حذف المكرر

print(f"📗 ratings_clean : {ratings_clean.shape}  | NaN: {ratings_clean.isnull().sum().sum()}")
print(f"📘 movies_clean  : {movies_clean.shape}  | NaN: {movies_clean.isnull().sum().sum()}")
print(f"📙 final_df      : {final_df.shape}  | NaN: {final_df.isnull().sum().sum()}")


FEATURE ENGINEERING & MERGE
📗 ratings_clean : (105334, 12)  | NaN: 0
📘 movies_clean  : (10323, 26)  | NaN: 0
📙 final_df      : (105334, 36)  | NaN: 0


In [85]:
import sqlite3
os.makedirs('../data/processed', exist_ok=True)

print("\n" + "="*60)
print("STRUCTURED STORAGE")
print("="*60)

# 1. Save to CSV
ratings_clean.to_csv('../data/processed/ratings_clean.csv', index=False)
movies_clean.to_csv('../data/processed/movies_clean.csv', index=False)
final_df.to_csv('../data/processed/final_df.csv', index=False)
print("✅ Saved to CSV")

# 2. Save to SQLite
conn = sqlite3.connect('../data/processed/movielens.db')
ratings_clean.to_sql('ratings_clean', conn, if_exists='replace', index=False)
movies_clean.to_sql('movies_clean', conn, if_exists='replace', index=False)
final_df.to_sql('final_df', conn, if_exists='replace', index=False)
conn.close()
print("✅ Saved to SQLite (movielens.db)")


STRUCTURED STORAGE
✅ Saved to CSV
✅ Saved to SQLite (movielens.db)


In [86]:
print("\n" + "="*60)
print("PYSPARK DEMONSTRATION")
print("="*60)

try:
    from pyspark.sql import SparkSession
    import pyspark.sql.functions as F
    
    # 1. Init Spark
    spark = SparkSession.builder.appName("MovieLens_Cleaning").master("local[*]").getOrCreate()
    print(f"✅ Spark version: {spark.version}")
    
    # 2. Load Data
    movies_spark = spark.read.csv('../data/raw/movies.csv', header=True, inferSchema=True)
    ratings_spark = spark.read.csv('../data/raw/ratings.csv', header=True, inferSchema=True)
    print(f"📊 Movies (Spark): {movies_spark.count()} rows")
    print(f"📊 Ratings (Spark): {ratings_spark.count()} rows")
    
    # 3. Quick Demo: Top 5 Movies by Average Rating
    print("\n📊 Top 5 Movies (Spark - Avg Rating with >= 50 ratings):")
    top_movies = ratings_spark.groupBy("movieId") \
        .agg(F.mean("rating").alias("avg_rating"), F.count("rating").alias("num_ratings")) \
        .filter(F.col("num_ratings") >= 50) \
        .orderBy(F.desc("avg_rating")) \
        .limit(5)
    top_movies.show()
    
    # 4. Stop Spark
    spark.stop()
    print("✅ PySpark demonstration completed successfully.")
    
except ImportError:
    print("⚠️ PySpark غير مثبت (pip install pyspark).")
    print("💡 تم تخطي هذا الجزء، لكن الكود جاهز للتشغيل.")


PYSPARK DEMONSTRATION
✅ Spark version: 4.2.0
📊 Movies (Spark): 10329 rows
📊 Ratings (Spark): 105339 rows

📊 Top 5 Movies (Spark - Avg Rating with >= 50 ratings):
+-------+-----------------+-----------+
|movieId|       avg_rating|num_ratings|
+-------+-----------------+-----------+
|    318|4.454545454545454|        308|
|    923|4.396103896103896|         77|
|    858|4.392857142857143|        210|
|   3000|4.384615384615385|         52|
|    904|4.331081081081081|         74|
+-------+-----------------+-----------+

✅ PySpark demonstration completed successfully.


In [87]:
print("\n" + "="*60)
print("BEFORE vs AFTER")
print("="*60)

summary = pd.DataFrame({
    'Metric': ['Ratings', 'Users', 'Movies', 'Movies rows'],
    'Before': [len(ratings_raw), ratings_raw['userId'].nunique(), ratings_raw['movieId'].nunique(), len(movies_raw)],
    'After' : [len(final_df), final_df['userId'].nunique(), final_df['movieId'].nunique(), len(movies)],
})
summary['Lost %'] = ((summary['Before'] - summary['After']) / summary['Before'] * 100).round(2)
display(summary)


BEFORE vs AFTER


,Metric,Before,After,Lost %
0,Ratings,105339,105334,0.00
1,Users,668,668,0.00
2,Movies,10325,10323,0.02
3,Movies rows,10329,10323,0.06
